# Solar Filament Segmentation — GPU Training (Ready-to-run)

**Objective:** Train a new U-Net challenger on the official Solar Filament Segmentation Challenge 2026 data. This notebook trains a new experimental challenger from scratch using our pinned research pipeline.

**Before running:** In Kaggle, select **GPU T4 / P100**, enable **Internet** (for the pinned MIT source), and attach competition data **filament-segmentation-2026**. Then select **Save Version → Save & Run All**.

**Training:** 20 epochs, 140 steps/epoch, batch size 4, 0.75× scale, boundary-aware BCE weight 3.0, month-separated training / tuning / untouched confirmation. The training objective is experimental; there is no claim it will outperform a previous checkpoint.

**Outputs:** `/kaggle/working/solar_auto14/day_05/training/best.pt`, `split.json`, `postprocess.json`, and `/kaggle/working/auto14_result.json`. Use the checkpoint only after the run finishes with `status=COMPLETE`; tuning-local PQ is **not** an official leaderboard score.

**Integrity:** Private Kaggle notebook, pinned and attributed MIT upstream, no test-label access, exports and validates submission.csv without submitting it automatically. Do not overwrite existing day-two checkpoint.


In [ ]:
"""Fail-closed, pinned-source enhancements for solar filament PQ research.

The upstream project is MIT licensed and pinned by the notebook builder.
These source-to-source modifications are *training-only*: inference checkpoints
remain compatible with the original upstream U-Net and loader. No test data,
leaderboard results, or reserved confirmation labels enter model selection.
"""
from __future__ import annotations

import ast


def _replace_exactly_once(source: str, needle: str, replacement: str, label: str) -> str:
    if source.count(needle) != 1:
        raise ValueError(f"Pinned {label} source has drifted; refusing to patch")
    return source.replace(needle, replacement)


def patch_auto14_postprocess_source(source: str) -> str:
    """Optionally reconnect aligned components; zero gap = upstream identity.

    Modifies only the postprocess layer. Both daily notebook and final notebook
    must apply this identical patch before importing solarseg.postprocess.
    """
    source = _replace_exactly_once(
        source, "import numpy as np\n",
        "import numpy as np\n\nfrom .auto14_geometry import link_aligned_fragments\n",
        "postprocess geometry import",
    )
    source = _replace_exactly_once(
        source, "    merge_dist: int = 0\n",
        "    merge_dist: int = 0\n    link_gap: int = 0\n",
        "postprocess configuration",
    )
    source = _replace_exactly_once(
        source, "    return filter_components(raw, cfg.min_area, cfg.max_instances)\n",
        """    result = filter_components(raw, cfg.min_area, cfg.max_instances)
    if cfg.link_gap:
        return link_aligned_fragments(result, max_gap=cfg.link_gap)
    return result
""",
        "postprocess instance recovery",
    )
    ast.parse(source, filename="solarseg/postprocess.py")
    return source


def patch_auto14_sources(train: str, losses: str, tune: str, postprocess: str | None = None) -> dict[str, str]:
    """Introduce boundary-aware BCE and month-balanced local hyperparameter tuning.

    Boundary emphasis is optional and equals the original upstream loss
    *exactly* when SOLAR_BOUNDARY_WEIGHT=0. The Dice term is unchanged.
    Sample selection is deterministic and spans available validation months.
    """
    train = _replace_exactly_once(
        train,
        "train_stems, val_stems = split_stems(stems, args.val_frac, args.seed)",
        "development_stems, confirmation_stems = split_stems(stems, 0.15, 2026)\n"
        "        train_stems, val_stems = split_stems(development_stems, 0.18, 2027)",
        "train month split",
    )
    train = _replace_exactly_once(
        train,
        'json.dumps({"train": train_stems, "val": val_stems}, indent=1)',
        'json.dumps({"train": train_stems, "val": val_stems, "confirmation": confirmation_stems}, indent=1)',
        "train split ledger",
    )
    losses = _replace_exactly_once(
        losses,
        "import torch\n",
        "import os\nimport torch\n",
        "loss imports",
    )
    losses = _replace_exactly_once(
        losses,
        "        self.dice_weight = dice_weight\n",
        '        self.dice_weight = dice_weight\n'
        '        self.boundary_weight = float(os.environ.get("SOLAR_BOUNDARY_WEIGHT", "0"))\n'
        '        if not 0 <= self.boundary_weight <= 8:\n'
        '            raise ValueError("Boundary weight must be between 0 and 8")\n',
        "loss initializer",
    )
    losses = _replace_exactly_once(
        losses,
        '        if not 0 <= self.boundary_weight <= 8:\n'
        '            raise ValueError("Boundary weight must be between 0 and 8")\n',
        '        if not 0 <= self.boundary_weight <= 8:\n'
        '            raise ValueError("Boundary weight must be between 0 and 8")\n'
        '        self.topology_weight = float(os.environ.get("SOLAR_TOPOLOGY_WEIGHT", "0"))\n'
        '        if not 0 <= self.topology_weight <= 0.5:\n'
        '            raise ValueError("Topology weight must be between 0 and 0.5")\n',
        "loss topology initializer",
    )
    losses = _replace_exactly_once(
        losses,
        "class BCEDiceLoss(nn.Module):\n",
        "\ndef _soft_erode(x):\n    # Directional 3x1 / 1x3 erosion protects long thin structures.\n    return torch.minimum(\n        -F.max_pool2d(-x, (3, 1), stride=1, padding=(1, 0)),\n        -F.max_pool2d(-x, (1, 3), stride=1, padding=(0, 1)),\n    )\n\n\ndef _soft_skeleton(x, iterations=3):\n    # Differentiable approximation; compute on TRAINING pixels only.\n    opened = F.max_pool2d(_soft_erode(x), 3, stride=1, padding=1)\n    skeleton = F.relu(x - opened)\n    for _ in range(iterations):\n        x = _soft_erode(x)\n        opened = F.max_pool2d(_soft_erode(x), 3, stride=1, padding=1)\n        detail = F.relu(x - opened)\n        skeleton = skeleton + F.relu(detail - skeleton * detail)\n    return skeleton\n\n\ndef soft_cldice_loss(logits, target):\n    \"\"\"Soft skeleton connectivity; bounded and differentiable for empty masks.\"\"\"\n    probability = torch.sigmoid(logits.float())\n    soft_target = target.float().clamp(0.0, 1.0)\n    pred_skeleton = _soft_skeleton(probability)\n    target_skeleton = _soft_skeleton(soft_target)\n    eps = 1.0\n    precision = ((pred_skeleton * soft_target).sum() + eps) / (pred_skeleton.sum() + eps)\n    sensitivity = ((target_skeleton * probability).sum() + eps) / (target_skeleton.sum() + eps)\n    return 1.0 - (2.0 * precision * sensitivity) / (precision + sensitivity + 1e-6)\n\n\n" + "class BCEDiceLoss(nn.Module):\n",
        "loss topology helper",
    )
    losses = _replace_exactly_once(
        losses,
        "        bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pos_weight)\n",
        """        if self.boundary_weight == 0:
            bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pos_weight)
        else:
            # Emphasize filament boundaries without erasing thin filaments.
            # The target is a soft multi-annotator consensus; do not binarize it.
            with torch.no_grad():
                upper = F.max_pool2d(target, kernel_size=7, stride=1, padding=3)
                lower = -F.max_pool2d(-target, kernel_size=7, stride=1, padding=3)
                edge = (upper - lower).clamp(0.0, 1.0)
                weight = 1.0 + self.boundary_weight * edge
            pixel_loss = F.binary_cross_entropy_with_logits(
                logits, target, pos_weight=self.pos_weight, reduction="none"
            )
            bce = (pixel_loss * weight).sum() / weight.sum()
""",
        "loss forward",
    )
    losses = _replace_exactly_once(
        losses,
        "        return bce + self.dice_weight * soft_dice_loss(logits, target)\n",
        "        total = bce + self.dice_weight * soft_dice_loss(logits, target)\n"
        "        if self.topology_weight:\n"
        "            total = total + self.topology_weight * soft_cldice_loss(logits, target)\n"
        "        return total\n",
        "loss topology contribution",
    )
    tune = _replace_exactly_once(
        tune,
        "    if args.max_images:\n        val_stems = val_stems[: args.max_images]\n",
        """    if args.max_images:
        # Head-of-list sampling overrepresents early observation times.
        # Deterministic round-robin across ALL tuning months improves coverage.
        from collections import defaultdict
        from random import Random
        months = defaultdict(list)
        for stem in val_stems:
            months[stem[:6]].append(stem)
        rng = Random(2027)
        for month in sorted(months):
            rng.shuffle(months[month])
        sampled = []
        while len(sampled) < min(args.max_images, len(val_stems)):
            progressed = False
            for month in sorted(months):
                if len(sampled) >= args.max_images:
                    break
                if months[month]:
                    sampled.append(months[month].pop())
                    progressed = True
            if not progressed:
                break
        val_stems = sorted(sampled)
""",
        "tune balanced month sampler",
    )
    revised = {"scripts/train.py": train, "solarseg/losses.py": losses, "scripts/tune.py": tune}
    if postprocess is not None:
        revised["solarseg/postprocess.py"] = patch_auto14_postprocess_source(postprocess)
    for path, text in revised.items():
        ast.parse(text, filename=path)
    return revised


import hashlib, json, os, subprocess, sys, time
from pathlib import Path
import torch
PLAN = {"day":5,"scale":0.75,"base":32,"epochs":20,"steps":140,"batch":4,"pos_weight":3,"pos_frac":0.75,"seed":137,"tuning_seed":2027,"confirmation_seed":2026,"holdout_fraction":0.15,"tuning_fraction_of_development":0.18,"model":"Farnoosh-Memari MIT U-Net, modified fixed month-split research challenger; optional edge-weighted BCE research arm","comparison":"tuning-local PQ; NO official score; no hidden test labels","boundary_weight":3}
REPO = "https://github.com/FarnooshMemari/solar-filament-segmentation.git"
REVISION = "ef37d2e93d4cc9e248595e791ee39fe520d392d0"
GEOMETRY_SOURCE = "\"\"\"Conservative directional reconnection of broken filament instances.\n\nAn instance can consist of several disconnected islands in COCO RLE. Grouping\nvisually aligned islands without *painting* a bridge avoids artificial positive\npixels and addresses the fragmentation penalty of Panoptic Quality.\n\nAll thresholds are full-resolution pixels. This module is purely geometric:\nit never sees annotation targets, hidden test labels, or Kaggle leaderboard.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\n\n\ndef _tips(labels: np.ndarray, ident: int, *, min_area: int, min_elongation: float):\n    ys, xs = np.nonzero(labels == ident)\n    if len(ys) < min_area:\n        return []\n    # Deterministic subsampling bounds PCA work for very long filament bodies.\n    if len(xs) > 20000:\n        indices = np.linspace(0, len(xs) - 1, num=20000, dtype=np.intp)\n        xs, ys = xs[indices], ys[indices]\n    pixels = np.stack((xs, ys), axis=1).astype(np.float64)\n    center = pixels.mean(axis=0)\n    centered = pixels - center\n    covariance = (centered.T @ centered) / max(len(pixels), 1)\n    values, vectors = np.linalg.eigh(covariance)\n    if values[-1] <= 0 or math.sqrt((values[-1] + 1e-6)/(values[0] + 1e-6)) < min_elongation:\n        return []\n    axis = vectors[:, -1]\n    projections = centered @ axis\n    # Robustly estimate the physical tip using the end 2% of filament pixels.\n    low = np.quantile(projections, 0.02)\n    high = np.quantile(projections, 0.98)\n    if high - low < 4:\n        return []\n    return [(ident, pixels[projections <= low].mean(axis=0), -axis),\n            (ident, pixels[projections >= high].mean(axis=0), axis)]\n\n\ndef link_aligned_fragments(\n    labels: np.ndarray,\n    *,\n    max_gap: int,\n    min_area: int = 80,\n    min_elongation: float = 2.5,\n    min_alignment: float = 0.90,\n) -> np.ndarray:\n    \"\"\"Link near-collinear filament tips, without changing any foreground pixel.\n\n    Greedy sorted endpoint matching allows one partner per tip. Components with\n    round/noisy shapes are excluded by minimum elongation and area. A zero gap\n    produces the exact original instance array. The newly assigned labels are\n    deterministic and compact; both source objects remain present geometrically.\n    \"\"\"\n    if not isinstance(labels, np.ndarray) or labels.ndim != 2 or labels.dtype.kind not in \"iu\":\n        raise TypeError(\"Expected a 2D integer instance label map\")\n    if np.any(labels < 0):\n        raise ValueError(\"Instance ids must be nonnegative\")\n    if type(max_gap) is not int or not 0 <= max_gap <= 80:\n        raise ValueError(\"max_gap must be an integer between 0 and 80\")\n    if type(min_area) is not int or min_area < 1:\n        raise ValueError(\"min_area must be a positive integer\")\n    if not (1 < min_elongation <= 20 and 0 < min_alignment <= 1):\n        raise ValueError(\"Invalid direction or elongation limits\")\n    ids = np.unique(labels)\n    ids = ids[ids != 0]\n    if max_gap == 0 or len(ids) <= 1:\n        return labels.copy()\n\n    endpoints = []\n    for ident in ids:\n        endpoints.extend(_tips(labels, int(ident), min_area=min_area,\n                               min_elongation=min_elongation))\n    candidates = []\n    for a, (ia, pa, ua) in enumerate(endpoints):\n        for b in range(a + 1, len(endpoints)):\n            ib, pb, ub = endpoints[b]\n            if ia == ib:\n                continue\n            delta = pb - pa\n            distance = float(np.linalg.norm(delta))\n            if not 0 < distance <= max_gap:\n                continue\n            direction = delta / distance\n            first = float(np.dot(ua, direction))\n            second = float(np.dot(ub, -direction))\n            if first < min_alignment or second < min_alignment:\n                continue\n            # Proximity wins, then larger alignment, then stable component ids.\n            cost = distance + max_gap * ((1-first) + (1-second))\n            candidates.append((cost, int(ia), int(ib), a, b))\n    candidates.sort()\n    parents = {int(x): int(x) for x in ids}\n\n    def root(x):\n        while parents[x] != x:\n            parents[x] = parents[parents[x]]\n            x = parents[x]\n        return x\n\n    used = set()\n    linked = 0\n    for _, ia, ib, a, b in candidates:\n        if a in used or b in used:\n            continue\n        ra, rb = root(ia), root(ib)\n        if ra == rb:\n            continue\n        parents[max(ra, rb)] = min(ra, rb)\n        used.update((a, b))\n        linked += 1\n    if not linked:\n        return labels.copy()\n\n    roots = {int(ident): root(int(ident)) for ident in ids}\n    canonical = {k: i + 1 for i, k in enumerate(sorted(set(roots.values())))}\n    replacement = {ident: canonical[rt] for ident, rt in roots.items()}\n    result = np.zeros_like(labels, dtype=np.int32)\n    for ident, new_id in replacement.items():\n        result[labels == ident] = new_id\n    return result\n"
GEOMETRY_TUNER_SOURCE = "#!/usr/bin/env python\n\"\"\"Tune conservative filament-tip linking on a small held-in month-balanced slice.\n\nRuns only AFTER the original threshold/area/dilation grid has been frozen.\nThe baseline and all three geometry settings see the exact same observations.\nConfirmation months, test images, and official Kaggle scores are never accessed.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import defaultdict\nimport json\nfrom pathlib import Path\nfrom random import Random\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\nimport numpy as np\n\nfrom solarseg.data import group_by_stem, load_records, record_instance_rles, resolve_paths\nfrom solarseg.infer import get_device, load_checkpoint, predict_image\nfrom solarseg.metrics import PQStats, match\nfrom solarseg.postprocess import PostConfig, instances_from_prob\nfrom solarseg.auto14_geometry import link_aligned_fragments\nfrom solarseg.rle import counts_to_rle, labels_to_counts\n\n\ndef balanced_sample(stems, maximum=24):\n    buckets=defaultdict(list)\n    for stem in stems:\n        buckets[stem[:6]].append(stem)\n    rng=Random(2027)\n    for month in sorted(buckets):\n        rng.shuffle(buckets[month])\n    chosen=[]\n    while len(chosen)<min(maximum,len(stems)):\n        progressed=False\n        for month in sorted(buckets):\n            if len(chosen)>=maximum:break\n            if buckets[month]:\n                chosen.append(buckets[month].pop())\n                progressed=True\n        if not progressed:break\n    return sorted(chosen)\n\n\ndef evaluate_one(labels, truth):\n    h,w=labels.shape\n    predicted=[counts_to_rle(c,h,w) for c in labels_to_counts(labels)]\n    stats=PQStats()\n    for gt in truth:\n        stats+=match(predicted,gt)\n    return stats\n\n\ndef run(run_dir: Path, max_images=24, gaps=(0,16,32,48)):\n    config=PostConfig.load(run_dir/\"postprocess.json\")\n    if config.link_gap!=0:\n        raise ValueError(\"Geometry tuner expects an untouched baseline postprocess config\")\n    split=json.loads((run_dir/\"split.json\").read_text())\n    train,validation,confirmation=(set(split[k]) for k in (\"train\",\"val\",\"confirmation\"))\n    if train & validation or train & confirmation or validation & confirmation:\n        raise RuntimeError(\"Data leakage among split partitions\")\n    month_sets=({stem[:6] for stem in partition} for partition in (train,validation,confirmation))\n    m_train,m_val,m_confirm=month_sets\n    if m_train&m_val or m_train&m_confirm or m_val&m_confirm:\n        raise RuntimeError(\"Month leakage among split partitions\")\n    selected=balanced_sample(sorted(validation),max_images)\n    if not selected:\n        raise RuntimeError(\"No development-only validation images to score\")\n    paths=resolve_paths()\n    records,_=load_records(paths.annotations,paths.train_images)\n    grouped=group_by_stem(records)\n    if not set(selected).issubset(grouped):\n        raise RuntimeError(\"Tuner validation samples missing from loaded labels\")\n    device=get_device(\"cuda\")\n    model,cfg=load_checkpoint(run_dir/\"best.pt\",device)\n    totals={int(g):PQStats() for g in gaps}\n    months={int(g):defaultdict(PQStats) for g in gaps}\n    for stem in selected:\n        anns=grouped[stem]\n        prob,disk=predict_image(\n            model,anns[0].image_path,device,cfg.get(\"scale\",0.5),\n            tile=cfg.get(\"tile\",512),overlap=cfg.get(\"overlap\",128),tta=False\n        )\n        original=instances_from_prob(prob,disk,config)\n        references=[record_instance_rles(ann) for ann in anns]\n        for gap in gaps:\n            labels=original if not gap else link_aligned_fragments(original,max_gap=int(gap))\n            stats=evaluate_one(labels,references)\n            totals[int(gap)]+=stats\n            months[int(gap)][stem[:6]]+=stats\n\n    baseline=totals[0].pq\n    # Minimum practical gain and plurality of monthly improvements protect\n    # against adopting a split/merge tweak from just one cherry-picked image.\n    needed=(len(m_val)+1)//2\n    candidates=[]\n    for gap in gaps:\n        if not gap:continue\n        improved=sum(months[int(gap)][m].pq>months[0][m].pq+1e-9 for m in\n                     sorted({s[:6] for s in selected}))\n        candidate=totals[int(gap)].pq\n        candidates.append((int(gap),candidate,improved))\n    allowed=[row for row in candidates if row[1]>=baseline+0.015\n             and row[2]>=min(needed,len({s[:6] for s in selected}))]\n    selected_gap,best_pq,better_months=(max(allowed,key=lambda x:(x[1],-x[0]))\n                                         if allowed else (0,baseline,0))\n    # Keep the detailed result in the same postprocess.json that the final\n    # notebook reads. All gains are *local tuning* and unconfirmed.\n    info=json.loads((run_dir/\"postprocess.json\").read_text()).get(\"validation\",{})\n    info[\"geometry_trial\"]={\n        \"baseline_local_pq\":float(baseline),\n        \"selected_local_pq\":float(best_pq),\n        \"link_gap\":int(selected_gap),\n        \"improved_validation_months\":int(better_months),\n        \"sample_images\":len(selected),\n        \"candidates\":[{\"gap\":g,\"pq\":float(pq),\"improved_months\":n}\n                      for g,pq,n in candidates],\n        \"metric\":\"LOCAL matching proxy; same development-only labels for selection\",\n    }\n    if selected_gap:\n        config.link_gap=int(selected_gap)\n        info[\"pq\"]=float(best_pq)\n    config.save(run_dir/\"postprocess.json\",extra=info)\n    return info[\"geometry_trial\"]\n\n\nif __name__==\"__main__\":\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\"--run\",required=True,type=Path)\n    parser.add_argument(\"--max-images\",type=int,default=24)\n    args=parser.parse_args()\n    print(json.dumps(run(args.run,args.max_images),sort_keys=True,indent=2))\n"
DAY = PLAN["day"]
ROOT = Path("/kaggle/working/solar_auto14")
TARGET = ROOT / ("day_%02d" % DAY)
RESULT = Path("/kaggle/working/auto14_result.json")
result = {"status":"STARTED","day":DAY,"training":True,"official_submission":False,
          "test_inference":False,"source_sha":REVISION,"config":PLAN,"research_only":True}
started = time.perf_counter()
try:
    assert torch.cuda.is_available(), "CUDA required for scheduled GPU training"
    assert not TARGET.exists(), "Refuse to overwrite prior experiment artifacts"
    TARGET.mkdir(parents=True, exist_ok=False)
    src = ROOT / "farnoosh_mit_reference"
    subprocess.run(["git","clone","-q",REPO,str(src)],check=True)
    subprocess.run(["git","-C",str(src),"checkout","-q","--detach",REVISION],check=True)
    actual = subprocess.check_output(["git","-C",str(src),"rev-parse","HEAD"],text=True).strip()
    assert actual == REVISION, "Pinned MIT source revision mismatch"
    licence = (src/"LICENSE").read_text()
    assert "MIT License" in licence and "Farnoosh Memari" in licence, "Third-party attribution missing"
    original_paths = ("scripts/train.py", "solarseg/losses.py", "scripts/tune.py", "solarseg/postprocess.py")
    originals = [(src/path).read_text() for path in original_paths]
    upgrades = patch_auto14_sources(*originals)
    provenance = {}
    for name in original_paths:
        revised = upgrades[name]
        (src/name).write_text(revised)
        provenance[name] = hashlib.sha256(revised.encode()).hexdigest()
    # Private third-party-compatible runtime extension; no original repository
    # source is copied into our repository. Geometric links NEVER paint pixels.
    geometry_path = src/"solarseg/auto14_geometry.py"
    geometry_path.write_text(GEOMETRY_SOURCE)
    tuner_path = src/"scripts/tune_geometry.py"
    tuner_path.write_text(GEOMETRY_TUNER_SOURCE)
    provenance["solarseg/auto14_geometry.py"] = hashlib.sha256(GEOMETRY_SOURCE.encode()).hexdigest()
    provenance["scripts/tune_geometry.py"] = hashlib.sha256(GEOMETRY_TUNER_SOURCE.encode()).hexdigest()
    (TARGET/"source_patches.json").write_text(json.dumps(provenance, sort_keys=True, indent=2)+"\n")
    result["source_patches_sha256"] = hashlib.sha256(
        json.dumps(provenance, sort_keys=True).encode()).hexdigest()
    result["boundary_weight"] = float(PLAN.get("boundary_weight", 0.0))
    result["topology_weight"] = float(PLAN.get("topology_weight", 0.0))
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "numpy==1.26.4","scipy==1.13.1",
                    "opencv-python-headless==4.10.0.84","pycocotools==2.0.11"],check=True)
    # Verify the actual compiled COCO extension in a NEW process, not merely
    # that import/installation succeeded. The Kaggle day-1 traceback ended in
    # pycocotools.mask.decode with missing NumPy macro NPY_OWNDATA.
    # The modern 2.0.11 wheel supports the current Python/NumPy C ABI.
    smoke = r"""
import numpy as np
from pycocotools import mask as coco
a=np.zeros((64,64),dtype=np.uint8)
a[12:17,30:35]=1
rle=coco.encode(np.asfortranarray(a))
assert int(coco.area(rle)) == 25
assert np.array_equal(coco.decode(rle),a)
assert np.array_equal(coco.decode(coco.merge([rle])),a)
"""
    subprocess.run([sys.executable,"-c",smoke],check=True)
    train_dir = TARGET / "training"
    cmd = [sys.executable, "scripts/train.py", "--out",str(train_dir),
           "--scale",str(PLAN["scale"]),"--base",str(PLAN["base"]),
           "--epochs",str(PLAN["epochs"]), "--steps-per-epoch",str(PLAN["steps"]),
           "--batch-size",str(PLAN["batch"]), "--workers","2",
           "--pos-weight",str(PLAN["pos_weight"]),
           "--pos-frac",str(PLAN["pos_frac"]), "--seed",str(PLAN["seed"]),
           "--val-every","5"]
    subprocess.run(cmd,cwd=src,check=True,
                   env={**os.environ,"SOLAR_BOUNDARY_WEIGHT":str(PLAN.get("boundary_weight",0.0)),
                        "SOLAR_TOPOLOGY_WEIGHT":str(PLAN.get("topology_weight",0.0))})
    split = json.loads((train_dir/"split.json").read_text())
    months = [{s[:6] for s in split[k]} for k in ("train","val","confirmation")]
    assert all(months) and not months[0]&months[1] and not months[0]&months[2] and not months[1]&months[2], "Month leakage or empty partitions"
    result["split_counts"] = {k:len(split[k]) for k in ("train","val","confirmation")}
    result["split_months_disjoint"] = True
    tune = [sys.executable,"scripts/tune.py","--run",str(train_dir),
            "--max-images","24",
            "--thresholds","0.35,0.45,0.55,0.65",
            "--min-areas","100,250,400,650", "--merge-dists","0,5,10"]
    subprocess.run(tune,cwd=src,check=True)
    # Evaluate the experimental orientation-constrained linker on the SAME
    # development-only validation months. If it fails, mark the entire run
    # incomplete rather than using a falsely high local model-selection PQ.
    if DAY in (5,7,9,11,13) and time.perf_counter()-started < 5400:
        geo = [sys.executable,"scripts/tune_geometry.py",
               "--run",str(train_dir),"--max-images","24"]
        subprocess.run(geo,cwd=src,check=True)
        result["geometry_tuning_attempted"] = True
    else:
        result["geometry_tuning_attempted"] = False
    post = json.loads((train_dir/"postprocess.json").read_text())
    local = post["validation"]
    pq = float(local["pq"])
    assert 0 <= pq <= 1 and pq == pq, "Local tuning PQ invalid"
    weight_path = train_dir / "best.pt"
    assert weight_path.exists() and weight_path.stat().st_size > 1024, "No checkpoint"
    sha = hashlib.sha256()
    with weight_path.open("rb") as handle:
        for piece in iter(lambda: handle.read(1024*1024),b""): sha.update(piece)
    result.update(status="COMPLETE",local_tuning_pq=pq,
                  tuning_pq_status="Exploratory: tuning and model selection reuse same validation",
                  best_checkpoint_sha256=sha.hexdigest(),
                  tuning_metric={k:local.get(k) for k in ("pq","sq","rq","images")},
                  checkpoint_output=str(weight_path),
                  confirmation_evaluated=False,
                  next_step="Preserve checkpoint for one independent final assessment")
except Exception as error:
    result.update(status="FAILED",error_type=type(error).__name__,
                  stage_error="Review private Kaggle logs; do not promote this candidate")
    raise
finally:
    result["elapsed_seconds"]=round(time.perf_counter()-started,1)
    RESULT.write_text(json.dumps(result,indent=2,sort_keys=True)+"\n")

## Generate and check the competition submission
After training completes, this cell predicts every test image and saves `/kaggle/working/submission.csv`. Download that CSV and upload it with **Submit Prediction → Upload Submission File**. The training notebook needs Internet and should not be submitted as an offline inference notebook.


In [ ]:
import json
import subprocess, sys
from pathlib import Path
result_file = Path('/kaggle/working/auto14_result.json')
if result_file.exists():
    training_report = json.loads(result_file.read_text())
    print('Training status:', training_report.get('status'))
    print('Local tuning PQ (NOT official):', training_report.get('local_tuning_pq'))
    print('Checkpoint SHA256:', training_report.get('best_checkpoint_sha256'))
    assert training_report['status'] == 'COMPLETE', 'Training failed; inspect preceding Kaggle logs'
    ckpt = Path(training_report['checkpoint_output'])
    assert ckpt.is_file() and ckpt.stat().st_size > 1024
    print('Saved trained checkpoint:', ckpt)
else:
    raise RuntimeError('Training output missing: check GPU, Internet, competition input and notebook logs')

# Export all official test images only after a completed training run.
submission_path = Path('/kaggle/working/submission.csv')
source_dir = Path('/kaggle/working/solar_auto14/farnoosh_mit_reference')
subprocess.run([sys.executable, 'scripts/predict.py', '--run', str(ckpt.parent),
                '--out', str(submission_path)], cwd=source_dir, check=True)
subprocess.run([sys.executable, 'scripts/check_submission.py', '--csv',
                str(submission_path)], cwd=source_dir, check=True)
summary = json.loads(submission_path.with_suffix('.summary.json').read_text())
assert summary['images'] > 0 and summary['rows'] > 0
training_report.update(test_inference=True, submission_validated=True,
                       submission_path=str(submission_path),
                       test_images=summary['images'], submission_rows=summary['rows'])
result_file.write_text(json.dumps(training_report, indent=2) + '\n')
print('Ready to upload using Submit Prediction:', submission_path)
